In [2]:
# -*- coding: utf-8 -*-
"""
茨城県 市町村パネルデータ + 隣接リストに基づいて
  - AR(1)モデル
  - AR(1) + 空間ラグ
  - AR(1) + 空間ラグ + SSDSE
を推定するスクリプト。

前提:
  panel_ibaraki_ssdse.csv  : これまでのスクリプトで作成済み
  ibaraki_neighbors.csv    : build_neighbors_from_shapefile.py で作成済み
"""

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error


# ======================================================
# 設定
# ======================================================

PANEL_FILE = "panel_ibaraki_ssdse.csv"
NEIGH_FILE = "ibaraki_neighbors.csv"

# SSDSE 説明変数として使う列（さっきまでの分析と合わせる）
# SSDSE 由来の「率」系＋世帯関連を使う
SSDSE_COLS = [
    "share_under15",   # 年少人口比率
    "share_over65",    # 高齢化率
    "nat_inc_rate",    # 自然増減率 = 出生率 - 死亡率
    "soc_inc_rate",    # 社会増減率 = 転入率 - 転出率
    "A7101",           # 世帯数（レベル情報）
    "A710101",         # 一般世帯数（レベル情報）
]



# ======================================================
# データ読み込み
# ======================================================

def load_panel(path=PANEL_FILE):
    print("=== Loading panel data ===")
    df = pd.read_csv(path)

    # city_name の前後空白除去
    df["city_name"] = df["city_name"].astype(str).str.strip()

    # 分母は時系列の人口
    denom = df["population"].replace(0, np.nan)

    # --- 既に入れているところ ---
    df["share_under15"] = df["A1301"] / denom
    df["share_over65"]  = df["A1303"] / denom
    df["birth_rate"]    = df["A4101"] / denom
    df["death_rate"]    = df["A4200"] / denom
    df["net_mig_rate"]  = (df["A5101"] - df["A5102"]) / denom

    # ★ ここから追加（自然増減率・社会増減率）★
    df["nat_inc_rate"] = df["birth_rate"] - df["death_rate"]
    df["soc_inc_rate"] = df["net_mig_rate"]

    df = df.sort_values(["city_name", "year"]).reset_index(drop=True)

    print("Panel shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head())
    return df


def load_neighbors(path=NEIGH_FILE):
    print("\n=== Loading neighbor list ===")
    neigh = pd.read_csv(path)
    neigh["city_name"] = neigh["city_name"].astype(str).str.strip()
    neigh["neighbor_name"] = neigh["neighbor_name"].astype(str).str.strip()
    print("Neighbors shape:", neigh.shape)
    print(neigh.head())

    # city -> [neighbors] の辞書に変換
    neighbors_dict = {}
    for row in neigh.itertuples(index=False):
        city = row.city_name
        nei = row.neighbor_name
        neighbors_dict.setdefault(city, set()).add(nei)

    # set -> list に変換
    neighbors_dict = {k: sorted(list(v)) for k, v in neighbors_dict.items()}
    print("\nNeighbor dict sample:")
    for i, (k, v) in enumerate(neighbors_dict.items()):
        print(" ", k, "->", v)
        if i >= 4:
            break

    return neighbors_dict


# ======================================================
# 空間ラグ項の作成
# ======================================================

def build_spatial_lag(panel: pd.DataFrame, neighbors_dict: dict):
    """
    各市町村 i, 年 t について、
        W_log_pop_lag1(i,t) = 隣接市町村 j の log_pop(i,t-1) の平均
    を作る。
    """
    print("\n=== Building spatial lag term W_log_pop_lag1 ===")

    # (city, year) -> log_pop の辞書
    key_to_log = {
        (row.city_name, int(row.year)): float(row.log_pop)
        for row in panel.itertuples(index=False)
    }

    years = sorted(panel["year"].unique())
    cities = sorted(panel["city_name"].unique())
    print("Years in panel:", years)
    print("Cities in panel:", cities)

    spatial_lag = []

    for row in panel.itertuples(index=False):
        city = row.city_name
        year = int(row.year)
        # モデルは log_pop_lag1 を使うので、空間ラグも t-1 の情報で作る
        target_year = year - 1

        neighs = neighbors_dict.get(city, [])
        vals = []

        for nei in neighs:
            val = key_to_log.get((nei, target_year), np.nan)
            if not np.isnan(val):
                vals.append(val)

        if len(vals) == 0:
            spatial_lag.append(np.nan)
        else:
            spatial_lag.append(float(np.mean(vals)))

    panel = panel.copy()
    panel["W_log_pop_lag1"] = spatial_lag

    print("\nPanel with W_log_pop_lag1 sample:")
    print(panel[["city_name", "year", "log_pop", "log_pop_lag1", "W_log_pop_lag1"]].head(15))

    return panel


# ======================================================
# モデル推定
# ======================================================

def split_train_test(panel: pd.DataFrame):
    """
    年で train / test を分ける。
    ここでは:
      - train: 年 <= 2010
      - test : 年 >= 2015
    とする（2011-2014はバッファ扱い）。
    """
    train_mask = panel["year"] <= 2010
    test_mask = panel["year"] >= 2015

    return train_mask, test_mask


def fit_ols(y, X, add_const=True, desc="model"):
    if add_const:
        X = sm.add_constant(X)

    model = sm.OLS(y, X, missing="drop")
    result = model.fit()
    print(f"\n===== OLS result: {desc} =====")
    print(result.summary())
    return result


def evaluate_predictions(y_true, y_pred, name="model"):
    mape = mean_absolute_percentage_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5  # squared=False がない古いsklearn対応
    print(f"\n{name} MAPE (test): {mape:.4f}")
    print(f"{name} RMSE  (test): {rmse:.2f}")
    return mape, rmse


def fit_models(panel: pd.DataFrame):
    print("\n=== Fitting models (AR, AR+W, AR+W+SSDSE) ===")

    # 欠損がある行を落とす（特に log_pop_lag1, W_log_pop_lag1, SSDSE）
    cols_needed = ["log_pop", "log_pop_lag1", "W_log_pop_lag1"] + SSDSE_COLS
    df = panel.copy()
    # SSDSE はあとで使うのでここでは全列がある必要はないが、
    # モデル3を回す都合で一括で dropna する。
    df = df.dropna(subset=["log_pop", "log_pop_lag1"])
    # 空間ラグを使うモデルでは W_log_pop_lag1 が NaN の行は落とす
    df_w = df.dropna(subset=["W_log_pop_lag1"]).copy()

    # train / test 分割
    train_mask_all, test_mask_all = split_train_test(df)
    train_mask_w, test_mask_w = split_train_test(df_w)

    # -------------------------
    # Model 1: AR(1) のみ
    # -------------------------
    y = df["log_pop"]
    X_ar = df[["log_pop_lag1"]]

    res_ar = fit_ols(y[train_mask_all], X_ar[train_mask_all], desc="AR(1)")

    # テスト予測
    X_test_ar = sm.add_constant(X_ar[test_mask_all])
    y_pred_ar = res_ar.predict(X_test_ar)
    y_true_ar = np.exp(y[test_mask_all])
    y_pred_ar_lin = np.exp(y_pred_ar)

    mape_ar, rmse_ar = evaluate_predictions(y_true_ar, y_pred_ar_lin, name="AR(1)")

    # -------------------------
    # Model 2: AR(1) + 空間ラグ
    # -------------------------
    y_w = df_w["log_pop"]
    X_ar_w = df_w[["log_pop_lag1", "W_log_pop_lag1"]]

    res_ar_w = fit_ols(y_w[train_mask_w], X_ar_w[train_mask_w], desc="AR(1)+W")

    X_test_ar_w = sm.add_constant(X_ar_w[test_mask_w])
    y_pred_ar_w = res_ar_w.predict(X_test_ar_w)
    y_true_ar_w = np.exp(y_w[test_mask_w])
    y_pred_ar_w_lin = np.exp(y_pred_ar_w)

    mape_ar_w, rmse_ar_w = evaluate_predictions(
        y_true_ar_w, y_pred_ar_w_lin, name="AR(1)+W"
    )

    # -------------------------
    # Model 3: AR(1) + 空間ラグ + SSDSE
    # -------------------------
    # SSDSE 列だけ抽出（存在する列だけに絞る）
    ssdse_cols_exist = [c for c in SSDSE_COLS if c in df_w.columns]
    print("\nUsing SSDSE columns:", ssdse_cols_exist)

    X_full = pd.concat(
        [
            df_w[["log_pop_lag1", "W_log_pop_lag1"]],
            df_w[ssdse_cols_exist],
        ],
        axis=1,
    )

    res_full = fit_ols(y_w[train_mask_w], X_full[train_mask_w], desc="AR(1)+W+SSDSE")

    X_test_full = sm.add_constant(X_full[test_mask_w])
    y_pred_full = res_full.predict(X_test_full)
    y_true_full = np.exp(y_w[test_mask_w])
    y_pred_full_lin = np.exp(y_pred_full)

    mape_full, rmse_full = evaluate_predictions(
        y_true_full, y_pred_full_lin, name="AR(1)+W+SSDSE"
    )

    # 結果まとめ
    results_table = pd.DataFrame(
        {
            "model": ["AR(1)", "AR(1)+W", "AR(1)+W+SSDSE"],
            "MAPE_test": [mape_ar, mape_ar_w, mape_full],
            "RMSE_test": [rmse_ar, rmse_ar_w, rmse_full],
        }
    )
    print("\n=== Summary of test errors ===")
    print(results_table)

    # 保存（発表資料用など）
    results_table.to_csv("spatial_model_error_summary.csv", index=False, encoding="utf-8-sig")
    print("\nSaved spatial_model_error_summary.csv")

    # 係数も CSV にしておくと便利
    coef_ar = res_ar.params.rename("AR(1)")
    coef_ar_w = res_ar_w.params.rename("AR(1)+W")
    coef_full = res_full.params.rename("AR(1)+W+SSDSE")

    coef_df = pd.concat([coef_ar, coef_ar_w, coef_full], axis=1)
    coef_df.to_csv("spatial_model_coefficients.csv", encoding="utf-8-sig")
    print("Saved spatial_model_coefficients.csv")

    return res_ar, res_ar_w, res_full


# ======================================================
# main
# ======================================================

def main():
    # 1) パネル読み込み
    panel = load_panel(PANEL_FILE)

    # 2) 隣接リスト読み込み
    neighbors_dict = load_neighbors(NEIGH_FILE)
    # neighbors_dict ができた後に追加でチェック
    for city, neighs in neighbors_dict.items():
        for nei in neighs:
            if city not in neighbors_dict.get(nei, []):
                print("WARNING: non-symmetric neighbor:", city, "<->", nei)
    # 3) 空間ラグ項 W_log_pop_lag1 を作成
    panel_with_W = build_spatial_lag(panel, neighbors_dict)
    panel_with_W.to_csv("panel_ibaraki_ssdse_with_W.csv", index=False, encoding="utf-8-sig")
    print("\nSaved panel_ibaraki_ssdse_with_W.csv")

    # 4) モデル推定
    fit_models(panel_with_W)


if __name__ == "__main__":
    main()


=== Loading panel data ===
Panel shape: (2068, 22)
Columns: ['city_name', 'year', 'population', 'log_pop', 'log_pop_lag1', 'city_code', 'A1301', 'A1302', 'A1303', 'A4101', 'A4200', 'A5101', 'A5102', 'A7101', 'A710101', 'share_under15', 'share_over65', 'birth_rate', 'death_rate', 'net_mig_rate', 'nat_inc_rate', 'soc_inc_rate']
  city_name  year  population    log_pop  log_pop_lag1  city_code   A1301  \
0   かすみがうら市  1976     36119.0  10.494574     10.485312        NaN  4376.0   
1   かすみがうら市  1977     36736.0  10.511512     10.494574        NaN  4376.0   
2   かすみがうら市  1978     37387.0  10.529078     10.511512        NaN  4376.0   
3   かすみがうら市  1979     37919.0  10.543208     10.529078        NaN  4376.0   
4   かすみがうら市  1980     38797.0  10.566098     10.543208        NaN  4376.0   

     A1302    A1303  A4101  ...   A5102    A7101  A710101  share_under15  \
0  22859.0  12779.0  176.0  ...  1239.0  15271.0  15230.0       0.121155   
1  22859.0  12779.0  176.0  ...  1239.0  15271.0  15230.0